# Colab — Stages 1–4: isotropy across quantizers, the GGUF ladder, η, and out-of-sample prediction

Runs the parts that do **not** fit on a 6 GB laptop GPU. See `docs/pivot_plan_2026-08.md` §5
and `docs/claims_and_evidence.md` §3.

| Stage | Claim | Why it needs Colab |
|---|---|---|
| **S1** | **C5** — isotropy is a property of the quantizer | needs AWQ/GPTQ builds + FP16 reference of a 3 B model |
| **S2** | **A3, C2** — measure η per scheme, **fit** the exponent | needs a matched GGUF ladder from one F16 checkpoint |
| **S3** | **C2** — d′ per scheme; η(weights) vs η(d′) | needs FP16 3 B (~6.4 GB) |
| **S4** | **C3** — fit on part of the ladder, predict the rest | cheap once S2/S3 exist |

**Gate.** Stage 0 (`stage0_noise_floor_and_isotropy.ipynb`) must PASS first. If the rotation
does not replicate across disjoint prompt halves, C1 is unproven and the geometric claims here
must be reported as conditional.

**Honest note on what is not here:** no completions are generated, so nothing in this notebook
supports a *behavioural* claim. That is Stage S6 (real generations + StrongREJECT), still to build.

## 0 — Environment and Drive checkpointing

In [ ]:
import os, sys, subprocess, pathlib, json, datetime

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/parnish007/CLIFFGUARD.git"
REPO_DIR = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _d; _d.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted (results will not survive a disconnect):", e)
    if not REPO_DIR.exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable,"-m","pip","-q","install",
                    "numpy<2","scipy","pydantic>=2","transformers","accelerate",
                    "bitsandbytes","datasets","gguf"], check=False)

if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
OUT = pathlib.Path("artifacts/colab_ladder"); OUT.mkdir(parents=True, exist_ok=True)

def checkpoint(name, payload):
    """Persist a stage result locally and to Drive so a disconnect costs at most one stage."""
    p = OUT / f"{name}.json"; p.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    if DRIVE.exists():
        d = DRIVE / "colab_ladder"; d.mkdir(parents=True, exist_ok=True)
        (d / f"{name}.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"[checkpoint] {p}")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if torch.cuda.is_available():
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))

In [ ]:
import numpy as np
from cliffguard.eval.noise_floor import difference_in_means, rotation_replication, angle_between
from cliffguard.eval.isotropy import isotropy_test
from cliffguard.eval.discriminability import d_prime_with_ci, gaussianity_gap, implied_eta
from cliffguard.eval.composition import (
    Composition, predict_collapse, collapse_bits_threshold_closed_form,
)

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # matches existing Fold A artifacts
LAYER    = 14
N_PER_CLASS = 200
SEED = 0
print("imports ok")

## 1 — Prompts (same corpus for every scheme — non-negotiable)

Every scheme must score the **identical prompt list in the identical order**. The paired design
depends on it: row *i* must be the same prompt everywhere, or `rotation_replication` and the
paired CI are both invalid.

In [ ]:
from pathlib import Path

def load_fold_a(n=N_PER_CLASS):
    for c in [Path("data/folds/fold_a"), Path("data/fold_a"), Path("notebooks/fold_a")]:
        jf = c / "fold_a.jsonl"
        if jf.exists():
            rows=[json.loads(l) for l in jf.read_text(encoding="utf-8").splitlines() if l.strip()]
            h=[r["prompt"] for r in rows if r.get("label")=="refused"][:n]
            b=[r["prompt"] for r in rows if r.get("label")=="benign"][:n]
            if len(h)>=4 and len(b)>=4:
                print(f"[prompts] {jf}: {len(h)} harmful / {len(b)} benign"); return h,b
    raise SystemExit(
        "Fold A corpus not found. Run scripts/download_fold_a.py first.\n"
        "Refusing to fall back to placeholder prompts — the results would be meaningless.")

harmful, harmless = load_fold_a()

## 2 — STAGE 1: is isotropy a property of the quantizer? (claim C5)

**The prediction.** RTN / NF4 / GGUF k-quants minimise unweighted reconstruction error, so nothing
couples their error to a behavioural direction → **isotropic** → cliff. AWQ / GPTQ explicitly
protect high-salience channels → **anisotropic** → no cliff.

If it holds it mechanistically resolves arXiv:2606.10154 (refusal falls 12–68 pp) versus
arXiv:2606.29581 (AWQ INT4 within ~1.6 pp of FP16 for 7 of 8 models).

**Decisive either way** — if AWQ is *also* isotropic, C5 is refuted and the literature split must
come from judges or corpora instead, which is equally worth reporting.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def loader_fp16(mid):
    return AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16,
                                                device_map="auto", output_hidden_states=True)
def loader_nf4(mid):
    return AutoModelForCausalLM.from_pretrained(mid, load_in_4bit=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
        device_map="auto", output_hidden_states=True)
def loader_awq(mid):
    # Point at a prebuilt AWQ repo, or build one with autoawq (Linux/Colab only).
    return AutoModelForCausalLM.from_pretrained(mid, device_map="auto", output_hidden_states=True)

SCHEMES = {"FP16": (MODEL_ID, loader_fp16), "NF4": (MODEL_ID, loader_nf4)}
# Add when you have an AWQ/GPTQ build of the SAME base checkpoint, e.g.:
# SCHEMES["AWQ_INT4"] = ("<org>/<Llama-3.2-3B-Instruct-AWQ>", loader_awq)

@torch.no_grad()
def collect(mid, loader, tag, prompts):
    cache = OUT / f"acts_{tag}_L{LAYER}_{len(prompts)}.npy"
    if cache.exists():
        print(f"[acts] cache hit {cache.name}"); return np.load(cache)
    tok = AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = loader(mid); model.eval()
    out=[]
    for i,p in enumerate(prompts):
        ids = tok.apply_chat_template([{"role":"user","content":p}],
                                      add_generation_prompt=True, return_tensors="pt").to(model.device)
        out.append(model(ids).hidden_states[LAYER][0,-1,:].float().cpu().numpy())
        if (i+1)%50==0: print(f"   {tag}: {i+1}/{len(prompts)}")
    arr=np.stack(out).astype(np.float64); np.save(cache, arr)
    del model; torch.cuda.empty_cache()
    return arr

acts={}
for tag,(mid,ld) in SCHEMES.items():
    acts[(tag,"h")] = collect(mid, ld, f"{tag}_h", harmful)
    acts[(tag,"l")] = collect(mid, ld, f"{tag}_l", harmless)
print({k:v.shape for k,v in acts.items()})

In [ ]:
dirs = {t: difference_in_means(acts[(t,"h")], acts[(t,"l")]) for t in SCHEMES}
iso_rows={}
for t in SCHEMES:
    if t=="FP16": continue
    # Stage 0 gate, per scheme
    rep = rotation_replication(acts[("FP16","h")], acts[("FP16","l")],
                               acts[(t,"h")], acts[(t,"l")], n_splits=50, seed=SEED)
    res = isotropy_test(dirs["FP16"], dirs[t], n_null=400, seed=SEED)
    print(f"===== {t} ====="); print(rep.summary()); print(res.summary()); print()
    iso_rows[t]={"replicates": rep.passes(), "replication_z": rep.z_score,
                 "angle_deg": res.angle_deg, "max_abs_z": res.max_abs_z,
                 "is_isotropic": res.is_isotropic(),
                 "irrecoverable_fraction": res.irrecoverable_fraction}
checkpoint("stage1_isotropy", iso_rows)
print("C5 is supported only if RTN/NF4/GGUF are isotropic AND AWQ/GPTQ are not.")
print("With NF4 alone this is a single data point, not a test of C5.")

## 3 — STAGE 2: matched GGUF ladder and η (claims A3, C2)

**Discipline that must not be relaxed:**
- every scheme from **one** F16 checkpoint (requantizing from a quantized file degrades quality),
- **one pinned** `llama.cpp` commit,
- NF4 / AWQ / GPTQ are a **separate categorical** comparison — never on the ordinal bit-width axis,
  because they are different algorithms and runtimes, not a monotone dose sequence,
- η is **fitted**, never assumed to be `4^(-b)`.

In [ ]:
LADDER = ["F16","Q8_0","Q6_K","Q5_K_M","Q4_K_M","Q3_K_M","Q2_K"]
LLAMA_CPP_COMMIT = ""   # PIN THIS. Record it in the manifest.

print("To build the ladder (once, then cache to Drive):")
print("  git clone https://github.com/ggml-org/llama.cpp && cd llama.cpp && git checkout <PIN> && make")
print("  python convert_hf_to_gguf.py <hf_model_dir> --outfile base-f16.gguf --outtype f16")
for q in LADDER[1:]:
    print(f"  ./llama-quantize base-f16.gguf model-{q}.gguf {q}")
print()
print("Then verify correspondence and peak memory BEFORE trusting any number:")
print("  python scripts/verify_gguf_pair.py base-f16.gguf model-Q4_K_M.gguf")
print()
print("NOTE: verify_gguf_pair.py is still marked UNVERIFIED-AGAINST-REAL-FILE.")
print("This run is the first opportunity to clear that marker — do it and report back.")

In [ ]:
# Measure eta per scheme from WEIGHTS (+ benign activations), then FIT the exponent.
from cliffguard.eval import noise_spectrum as ns

GGUF_DIR = pathlib.Path("/content/gguf")   # or a Drive path
eta_by_scheme, bits_by_scheme = {}, {}

if GGUF_DIR.exists():
    for q in LADDER[1:]:
        f16, qq = GGUF_DIR/"base-f16.gguf", GGUF_DIR/f"model-{q}.gguf"
        if not (f16.exists() and qq.exists()):
            print(f"[skip] {q}: missing file"); continue
        try:
            rep = ns.measure_gguf_pair(str(f16), str(qq), direction=dirs["FP16"])
            eta_by_scheme[q] = rep.eta_proxy      # PROXY — see docs/build_log.md Entry 4
            bits_by_scheme[q] = rep.bits_per_param_payload
            print(f"  {q}: eta_proxy={rep.eta_proxy:.4f}  payload bits/param={rep.bits_per_param_payload:.3f}")
        except Exception as e:
            print(f"[error] {q}: {type(e).__name__}: {e}")
else:
    print("No GGUF directory — build the ladder first (cell above).")

if len(eta_by_scheme) >= 3:
    fit = ns.fit_eta_vs_bits_report({bits_by_scheme[k]: v for k,v in eta_by_scheme.items()})
    print(); print(fit)
    print("\nIf the exponent CI EXCLUDES 4, assumption A3 fails for k-quants.")
    print("That is a RESULT, not a failure — Theorem 2 then uses the measured base.")
    checkpoint("stage2_eta", {"eta_proxy": eta_by_scheme, "bits": bits_by_scheme})

## 4 — STAGE 3: d′ per scheme, and the central validation (claim C2)

**The core test of the whole theory:** η measured from *weights* must agree with η implied by the
*d′ decay*. Agreement supports the mechanism; disagreement **refutes C2** (falsifier F5).

In [ ]:
def margins(A, r):
    r = r/np.linalg.norm(r); return (A @ r)/np.linalg.norm(A, axis=1)

dp={}
for t in SCHEMES:
    mh = margins(acts[(t,"h")], dirs[t]); ml = margins(acts[(t,"l")], dirs[t])
    np.save(OUT/f"margins_{t}_harmful.npy", mh)     # RAW margins — required for d'_0
    np.save(OUT/f"margins_{t}_benign.npy",  ml)
    d = d_prime_with_ci(mh, ml, fires_high=False, n_bootstrap=2000, seed=SEED)
    gap = gaussianity_gap(mh, ml, fires_high=False)
    dp[t] = {"d_prime": d.d_prime, "ci": [d.ci_low, d.ci_high], "gaussianity_gap": gap}
    print(f"{t:8s} {d.summary()}")
    print(f"         gaussianity gap = {gap:.3f}  "
          f"{'(A2 OK)' if gap<=0.05 else '(A2 FAILS — closed-form TPR predictions are void)'}")

for t in SCHEMES:
    if t=="FP16": continue
    eta_beh = implied_eta(dp["FP16"]["d_prime"], dp[t]["d_prime"])
    eta_w   = eta_by_scheme.get(t)
    print(f"\n{t}: eta from d' decay = {eta_beh:.4f}")
    if eta_w is not None:
        print(f"{t}: eta from weights  = {eta_w:.4f}   ratio = {eta_beh/eta_w:.2f}")
        print("     Ratio far from 1 => F5 fires => C2's mechanism is REFUTED. Report it.")
checkpoint("stage3_dprime", dp)

## 5 — STAGE 4: fit on part of the ladder, predict the rest (claim C3)

**The headline result.** Fit `d'_0` and `η₄` on the HIGH-precision end only, then predict collapse
at the low-precision end **without refitting**. In-sample fit proves nothing; only the out-of-sample
residual does.

In [ ]:
if len(eta_by_scheme) >= 4 and len(dp) >= 2:
    items = sorted(bits_by_scheme.items(), key=lambda kv: -kv[1])
    train = items[:len(items)//2]; test = items[len(items)//2:]
    print("train (high precision):", [k for k,_ in train])
    print("test  (low precision) :", [k for k,_ in test])
    d0 = dp["FP16"]["d_prime"]
    eta4_train = float(np.median([eta_by_scheme[k]*(4.0**(b-4.0)) for k,b in train]))
    b_star = collapse_bits_threshold_closed_form(d0, 0.05, eta4_train)
    print(f"\nfitted eta_4 (train only) = {eta4_train:.4f}")
    print(f"predicted collapse b*     = {b_star:.2f} bits")
    print("\nOUT-OF-SAMPLE CHECK — predicted vs actual d' on held-out schemes:")
    from cliffguard.eval.composition import d_prime_at_bits
    for k,b in test:
        pred = d_prime_at_bits(b, d0, eta4_train)
        act  = dp.get(k,{}).get("d_prime")
        print(f"  {k:8s} bits={b:5.2f}  predicted d'={pred:.3f}"
              + (f"  actual={act:.3f}  err={pred-act:+.3f}" if act else "  (no d' measured)"))
    checkpoint("stage4_prediction", {"eta4_train": eta4_train, "b_star": b_star,
                                     "train":[k for k,_ in train], "test":[k for k,_ in test]})
else:
    print("Need >=4 ladder points with eta and >=2 d' measurements. Complete S2/S3 first.")

## 6 — Provenance manifest (required before any result is quoted)

In [ ]:
import hashlib, subprocess as sp
def git_sha():
    try: return sp.check_output(["git","rev-parse","HEAD"], text=True).strip()
    except Exception: return "unknown"

manifest = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "git_sha": git_sha(),
    "model_id": MODEL_ID, "layer": LAYER, "n_per_class": N_PER_CLASS, "seed": SEED,
    "llama_cpp_commit": LLAMA_CPP_COMMIT or "UNPINNED — results not reproducible",
    "schemes": list(SCHEMES), "ladder": LADDER,
    "prompt_manifest_sha256": hashlib.sha256(
        ("\n".join(harmful+harmless)).encode("utf-8")).hexdigest(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
checkpoint("manifest", manifest)
print(json.dumps(manifest, indent=2))
if not LLAMA_CPP_COMMIT:
    print("\nWARNING: llama.cpp commit is unpinned. The ladder is not reproducible.")

---
## Reading the outcome

| Result | Meaning |
|---|---|
| Rotation does not replicate | **C1 unproven.** Everything geometric here is conditional. Report and stop. |
| NF4/GGUF isotropic, AWQ not | **C5 supported** — resolves the 2606.10154 / 2606.29581 split. |
| AWQ also isotropic | **C5 refuted.** The split comes from judges/corpora. Still publishable. |
| Exponent CI excludes 4 | **A3 fails for k-quants.** A result — Theorem 2 uses the measured base. |
| gaussianity gap > 0.05 | **A2 fails.** d′ and the ordering survive; drop the closed-form TPR claims. |
| η(weights) ≉ η(d′) | **F5 fires — C2's mechanism is refuted.** The most important negative outcome. |
| Out-of-sample prediction lands | **C2/C3 supported.** This is the paper. |

Every row is publishable. None of them is a reason to adjust the analysis after seeing the data —
pre-register first (`docs/claims_and_evidence.md` §4).